# Tutorial: OpenArcade's Bidding System example
- Author: Shridhar Kini
- To Securely Run: `jupyter notebook password` to generate onetime password for secure access
- To Run at root directory of the repo: `jupyter notebook --allow-root --port 9999 --ip=0.0.0.0`
- To Clear Outputs: Use `jupyter nbconvert --clear-output --inplace Bids_Processing.ipynb`

**github: https://github.com/opencyber-space/openarca.de**

**What this tutorial shows:** how OpenArcade runs a bid job.

* Buyer posts a job
* 5 manager teams are asked to bid
* A manager that cannot do the job declines
* OpenArcade picks the winner
* The winning team does the work with its sub-agents


**4 jobs:** security audit, React component, image edit, marketing article.

| Team | Manager | Sub-agents |
| :--- | :--- | :--- |
| **Security Audit** | `manager1-security-audit` | code scanner, dependency checker, compliance verifier |
| **Security Audit (solo)** | `manager5-security-audit` | 1 generalist doing all three jobs |
| **Frontend** | `manager2-frontend` | UI builder, state/API integrator, style designer |
| **Image Editing** | `manager3-image-editing` | enhancer, object segmenter, metadata/watermarker |
| **Content Creation** | `manager4-content-creation` | copywriter, SEO optimiser, proofreader |

**Scale:** 18 agent pods, 2 functions, 1 evaluator decision per job.

**Why two security teams:**
* Both can do the job
* One has 3 specialists, the other has 1 generalist
* The evaluator only reads the numbers on the bids

# The systems this example sits on

```
                     the buyer  (bids_inference_request.sh)
                          |  POST /bid-jobs   -- one job, five subjects named
                          v
                    OpenArcade  (the bidding system)
                          |  sends each subject a bid_request
                          |  runs the PQT function on every bid that arrives
                          |  runs the evaluator ONCE, when the last bid lands
                          |  sends the winner a bid_winner notification
                          v
                     AgentGrid  (the 18 agents)
                          |  OpenMesh carries manager <-> sub-agent messages
                          |  HIS records what every agent received and produced
```

| System | What it does here |
| :--- | :--- |
| **OpenArcade** | Runs the bid job: sends requests, checks bids, picks the winner |
| **ServiceGrid Functions** | 2 small functions: `dummy-pqt` accepts every bid, `bids-evaluator` picks the cheapest |
| **AgentGrid** | The 18 agents. Managers talk to their sub-agents over OpenMesh |
| **HIS** | Stores what each agent received and produced. The dashboard reads it |

**No marketplace layer here:**
* The buyer names the 5 managers directly
* `video_analytics_bidding` adds XChange in front of the same OpenArcade, so the buyer does not need to know every name

## 1. Directory Structure

* **`nodes/`** - agent code, one file per agent
  * Managers: bidding logic and implementation rounds
  * Sub-agents: handle `assess`, `implement`, `verify`
* **`spec/`** - 18 agent specs plus `register.sh` / `unregister.sh`
* **`functions/`** - `dummy-pqt` and `bids-evaluator`, with build / upload / call / delete scripts
* **`deploy_run/`** - this notebook, deploy and remove scripts, request script, image seeder, dashboard

**Job text lives in `bids_inference_request.sh`.** Edit it to change what the teams are asked. No agent code changes.

## 2. Prerequisites

1. `.env` at the repo root, created from `.env.template`
2. Cluster running: OpenArcade, function registry, agent deployer, NATS, HIS
3. `OPENAI_API_KEY` and an inference server (these are real LLM agents)
4. MinIO - only needed for the image job (section 11)

In [ ]:
%%bash
(
  cd ../../..
  set -a; source .env; set +a
  # Without these the round cannot leave this machine.
  for v in OPENARCADE_BIDDING_URL API_BASE_URL DELEGATE_API_URL NATS_URL \
           FUNCTION_REGISTRY_URL FUNCTION_UPLOAD_URL HIS_BASE_URL DOCKER_REGISTRY; do
    printf '  %-26s %s\n' "$v" "${!v:-*** REQUIRED, NOT SET ***}"
  done
  # Set, or not -- but never printed.
  for v in OPENAI_API_KEY MINIO_ACCESS_KEY MINIO_SECRET_KEY; do
    printf '  %-26s %s\n' "$v" "$([ -n "${!v:-}" ] && echo 'set' || echo '*** NOT SET ***')"
  done
)

## 3. How OpenArcade runs a bid job

The most important section. Everything else supports it.

```
  POST /bid-jobs                     the buyer opens a job and names the subjects
        |
        +--> bid_request  --------->  each named subject, in parallel
        |                              (task_results records one row per dispatch)
        |
        +<-- POST /bid-jobs/{id}/bids  each subject submits exactly one bid
        |         |
        |         +--> the PQT function runs on THAT bid, synchronously.
        |              Rejected? accepted=false -> bid_status=declined
        |
        +--> when EVERY subject has a bid on file:
        |         the evaluator function runs ONCE, on a background thread
        |
        +--> bid_winner ----------->  the winning subject only
```

**Evaluation Function Trigger:**
* Runs once, when bids from all subjects are received
* No deadline and no timeout
* A manager that fails to send a bid keeps evaluation on hold
* So every agent path, even a decline, must submit a bid

**REST calls:**

| Call | Use |
| :--- | :--- |
| `POST /bid-jobs` | Open the job, returns `bid_job_id` |
| `GET /bid-jobs/{id}/bids` | All bids, with `is_winner` |
| `GET /bid-jobs/{id}/task-results` | Who was asked, and who won |
| `POST /bids/query`, `POST /bid-jobs/query` | Search after the fact |

**The two functions OpenArcade calls:**

| Function | Gets | Returns |
| :--- | :--- | :--- |
| **PQT** (`bid_job_pqt_id`) | one bid | `{"accepted": true or false, "reason": ...}` |
| **Evaluator** (`bid_job_evaluator_id`) | all bids | `{"winner_subject_id": ..., "status": "resolved", "result_data": {...}}` |

**A decline is a normal bid:**
* Marked `bid_status: "declined"`
* Carries a huge token number (`9007199254740991`) so it can never win
* The evaluator also skips it by status

## 4. Build and upload the two functions

**`dummy-pqt`**
* Accepts every bid
* The place to add real pre-qualification checks

**`bids-evaluator`** - picks the cheapest bid

```python
score = (total_estimated_tokens * 0.5) + (required_compute * 0.5)   # lowest wins
```

* Compute words become numbers: `low` 500, `moderate` 1500, `high` 3000
* Declined bids are skipped

`build.sh` zips both functions (`function.json` + `function.zip`):

In [ ]:
%%bash
(
  cd ../functions
  bash build.sh
)

`upload.sh` deletes the old versions, then uploads both. Read 4.2 before re-uploading.

In [ ]:
%%bash
(
  cd ../functions
  bash upload.sh
)

### 4.1. Call the functions directly

`call.sh` runs each function once with sample input.

* No pods, no LLM, no bid job needed
* Sample evaluator input has 3 fake agents
* Expected winner: `agent-2` (80 tokens, 40 compute)

In [ ]:
%%bash
(
  cd ../functions
  bash call.sh all
)

### 4.2. Keep the version in sync

* Re-uploading the same function id keeps serving the old code
* Fix: bump `function_version` in the manifest
* `delete.sh` only removes the version named in the manifest

The cell below compares the registry, the manifest and the request script. All three should show the same version:

In [ ]:
%%bash
(
  cd ../../..
  set -a; source .env; set +a
  for id in bids-evaluator:1.0-stable bids-evaluator:1.1-stable dummy-pqt:1.0-stable; do
    printf '  %-28s ' "$id"
    curl -s --max-time 15 "$FUNCTION_REGISTRY_URL/functions/$id" |
      jq -r 'if (.function_id // empty) != "" then "in the registry" else "not found" end'
  done
)
echo ""
echo "  manifest says:  $(jq -r '.function_version + "-" + .function_release_tag' ../functions/bids-evaluator/function.json)"
echo "  the buyer asks: $(grep -o 'bids-evaluator:[0-9.]*-stable' bids_inference_request.sh | sort -u | tr '\n' ' ')"

## 5. Register the 18 agent specs

Managers find their sub-agents by tag, not by a fixed list:

```python
known_agents.query_and_add(query={"metadata.subject_search_tags": "security-subordinates"})
self.subordinates = [agent.id for agent in known_agents.list_all()]
```

* Security team 1: tag `security-subordinates` (3 specialists)
* Security team 2 (solo): tag `security-solo-subordinates` (1 generalist)
* A new agent with the right tag is picked up next round. No code change.

In [ ]:
%%bash
(
  cd ../spec
  bash register.sh
)

## 6. Build and push the agent images

* One image per agent, tagged with its `subject_id`
* Takes a while for all 18
* Filter options: `security`, `frontend`, `image`, `content`, `all`

In [ ]:
%%bash
(
  cd ../..
  bash build_bid_processing_agents_and_push.bash all
)

## 7. Deploy the 18 pods

In [ ]:
%%bash
(
  cd ../deploy_run
  bash deploy.sh
)

### Verify that the agents are running

* `Running` does not mean ready
* Give the agents a moment to join the mesh

In [ ]:
!kubectl get pods -n agents | grep -E 'NAME|manager|subagent' | head -25

## 8. Run a bidding round

`bids_inference_request.sh` acts as the buyer:
* Creates the job
* Waits for all bids
* Waits for the winner result

Pick jobs with flags at the top of the script, or override for one run:

| Flag | Job |
| :--- | :--- |
| `ONLY_SECURITY` | Security audit of a broken auth service (**on by default**) |
| `ONLY_FRONTEND` | React login form |
| `ONLY_IMAGE` | Hero image enhancement (needs MinIO, section 11) |
| `ONLY_CONTENT` | 300-word SEO article |
| `RUN_ALL` | All four, one after another |

**What happens in the security job:**
1. All 5 managers get the same request
2. Frontend, image and content managers decline (not their domain)
3. The 2 security managers ask their sub-agents, then combine the answers into one bid
4. Every bid goes through `dummy-pqt`
5. The 5th bid triggers the evaluator. Cheapest qualifying bid wins.
6. The winner starts the implementation phase (section 9)

### 8.1. The job payload

```json
{
  "bid_job_name":        {"en": "Security Audit Pass"},
  "bid_job_description": {
    "task_type": "security_review",
    "text": "... the job, with the files inlined ...",
    "acceptance_criteria": {
      "forbidden_pins":   ["Flask==1.1.2", "..."],
      "required_config":  {"session.cookie_secure": true, "...": "..."},
      "forbidden_code":   ["hardcoded-dev-secret"]
    }
  },
  "bid_job_evaluator_id": "bids-evaluator:1.1-stable",
  "bid_job_pqt_id":       "dummy-pqt:1.0-stable",
  "bid_job_creator_id":   "user",
  "bid_job_subject_ids":  ["manager1-security-audit", "...", "manager5-security-audit"]
}
```

**About `acceptance_criteria`:**
* OpenArcade does not read it
* The winner's work is checked against it later, in code (`measure_solution`)

In [ ]:
%%bash
(
  cd ../deploy_run
  ONLY_SECURITY=true bash bids_inference_request.sh
)

### 8.2. Read the round back from OpenArcade

The cell below finds the latest round and prints:
* the job
* every bid
* who was asked and who won

In [ ]:
%%bash
(
  cd ../../..
  set -a; source .env; set +a
  MANAGERS='["manager1-security-audit","manager2-frontend","manager3-image-editing","manager4-content-creation","manager5-security-audit"]'

  # Bid jobs carry no creation timestamp; bids do. So find the newest bid by one of these
  # five managers and take the job it belongs to.
  JOB=$(curl -s --max-time 60 -X POST "$OPENARCADE_BIDDING_URL/bids/query" -H 'Content-Type: application/json' -d "{\"bid_subject_id\": {\"\$in\": $MANAGERS}}" |
        jq -r '(.data // .) | sort_by(.submission_time) | last | .bid_job_id')
  echo "latest bid job: $JOB"

  curl -s --max-time 30 "$OPENARCADE_BIDDING_URL/bid-jobs/$JOB" |
    jq '(.data // .) | {name: .bid_job_name.en, task_type: .bid_job_description.task_type,
                        evaluator: .bid_job_evaluator_id, pqt: .bid_job_pqt_id,
                        asked: (.bid_job_subject_ids | length)}'

  echo ""
  echo "bids on file:"
  curl -s --max-time 30 "$OPENARCADE_BIDDING_URL/bid-jobs/$JOB/bids" |
    jq -r '(.data // .)[] |
           "  \((.bid_subject_id + "                              ") | .[0:30])status=\(.bid_data.bid_status // "?") tokens=\(.bid_data.total_estimated_tokens // "?") compute=\(.bid_data.required_compute // "?") winner=\(.is_winner)"'

  echo ""
  echo "dispatch trail:"
  curl -s --max-time 30 "$OPENARCADE_BIDDING_URL/bid-jobs/$JOB/task-results" |
    jq -r '(.data // .)[] | "  \(.task_type)  \(.bid_subject_id)"'
)

Sample result from a real round:

```
bids on file:
  manager2-frontend             status=declined  tokens=9007199254740991  winner=false
  manager4-content-creation     status=declined  tokens=9007199254740991  winner=false
  manager3-image-editing        status=declined  tokens=9007199254740991  winner=false
  manager5-security-audit       status=submitted tokens=2200  compute=moderate  winner=true
  manager1-security-audit       status=submitted tokens=2600  compute=moderate  winner=false
```

* Both security teams bid `moderate` compute
* The token estimate decides: 2,200 vs 2,600
* The solo team wins

## 9. What the winner does after winning

* The winner gets a short `bid_winner` message with only the `bid_job_id`
* The manager looks up its saved session for that job

Then up to 3 rounds (`MAX_APPROVAL_LOOPS = 3`):

1. **Route** - manager picks who implements and who reviews
2. **Implement** - sub-agents improve one shared solution
3. **Measure** - code checks the facts, not the model
   * Does the code parse
   * Are the vulnerable pins gone
   * Is the config set
   * Image job: `image_facts` reads real size, DPI and brightness
4. **Verify** - reviewer judges the result using those measured facts

## 10. Watch it in the dashboard

* Reads HIS
* One tab per team, one column per agent
* Shows messages between manager and sub-agents, in order
* Runs on port **8007**

In [ ]:
%%bash
(
  cd ../deploy_run
  nohup bash run_streamlit_app.bash > /tmp/bids_processing_streamlit.log 2>&1 &
  sleep 5
  tail -5 /tmp/bids_processing_streamlit.log
)

## 11. The image job

The other 3 jobs put their input in the payload. The image job needs a file in MinIO.

`seed_image.py`:
* Builds the hero banner (1280x720, 72 DPI, sRGB, underexposed, with EXIF)
* Uploads it to the `marketing-images` bucket
* Prints the `input_image` reference for the job
* Reuses the image next time. `SEED_FORCE=1` regenerates it.
* No API key: uses a drawn placeholder, the job still works

**Manual step to open the image URL in a browser:**

```
MinIO Console -> Buckets -> marketing-images -> Anonymous -> Add Access Rule
    Prefix:  /
    Access:  readonly
```

* Agents do not need this (they have MinIO credentials)
* A browser does. Without it the link shows 403.

In [ ]:
%%bash
(
  cd ../deploy_run
  ../../../venv/bin/python seed_image.py | jq '{bucket, object, url, facts: {width: .facts.width, height: .facts.height, dpi: .facts.dpi, avg_luminance: .facts.avg_luminance, exif_tags: .facts.exif_tags}}'
)

Run the image job on its own. It takes minutes:

In [ ]:
%%bash
(
  cd ../deploy_run
  ONLY_SECURITY=false ONLY_IMAGE=true RESULT_TIMEOUT=900 bash bids_inference_request.sh
)

## 12. Run the checks without a cluster

* Both functions have local tests
* No cluster needed
* Run them before changing the evaluator

In [ ]:
%%bash
(
  cd ../functions/bids-evaluator
  ../../../../venv/bin/python test.py
)
(
  cd ../functions/dummy-pqt
  ../../../../venv/bin/python test.py
)

## 13. Things that can go wrong in a live demo

* **Evaluation stuck at 4/5 bids:** one manager has not sent its bid. Evaluation waits for all subjects.
* **Function re-uploaded but old code still runs:** bump the version (see 4.2)
* **Manager with no sub-agents:** a wrong tag makes it bid alone, cheaply. Only the HIS timeline shows it.
* **`required_compute` comes back as text:** it must be `low`, `moderate` or `high`. `normalize_bid` fixes it. Do not remove it.
* **`Running` is not ready:** wait a moment after `deploy.sh`
* **Bid polling stops at 120s:** slow inference. The job is still live, only the script gave up.

## 14. Clean Up and Destroy Resources

Order: deployments, then specs, then functions. This way nothing is sent to an agent that no longer exists.

In [ ]:
%%bash
(
  cd ../deploy_run
  bash remove.sh
)

In [ ]:
%%bash
(
  cd ../spec
  bash unregister.sh
)

In [ ]:
%%bash
(
  cd ../functions
  bash delete.sh
)